# Bank Customer Complaint Analysis (UCI/Kaggle)

Notebook complet pour :

- Analyse exploratoire
- Prétraitement du texte
- Classification (TF-IDF + Naive Bayes / LogisticRegression)
- Visualisations

**Remarque** : Télécharge le dataset `bank_customer_complaint` (ou `Bank Customer Complaint Analysis`) depuis UCI / Kaggle et place le fichier CSV dans le même dossier que ce notebook. Les liens utiles :

- UCI Repository: https://archive.ics.uci.edu/
- Exemple Kaggle mirror: https://www.kaggle.com/datasets/adhamelkomy/bank-customer-complaint-analysis

---


In [ ]:
# Imports de base
import os
import pandas as pd
import numpy as np

# NLP & ML
import re
import string
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# Visualisation (matplotlib requis — pas seaborn)
import matplotlib.pyplot as plt

# Affichage pandas
pd.set_option('display.max_colwidth', 300)
pd.set_option('display.max_rows', 100)


## 1) Charger le dataset

Place le fichier CSV (ex: `BankCustomerComplaints.csv` ou `bank_complaints.csv`) dans le même dossier que ce notebook.

Exemples de méthodes pour charger :

```python
# si le fichier s'appelle bank_complaints.csv
df = pd.read_csv('bank_complaints.csv')
```

Si tu veux télécharger depuis Kaggle, installe `kaggle` CLI et utilise :

```bash
kaggle datasets download -d adhamelkomy/bank-customer-complaint-analysis
unzip bank-customer-complaint-analysis.zip
```


In [ ]:
# Exemple : charger un fichier local (adapte le nom si nécessaire)
file_candidates = ['bank_complaints.csv', 'BankCustomerComplaints.csv', 'BankCustomerComplaint.csv', 'BankCustomerComplaintAnalysis.csv', 'Bank_Customer_Complaint.csv']

for f in file_candidates:
    if os.path.exists(f):
        print('Loading', f)
        df = pd.read_csv(f, encoding='utf-8', low_memory=False)
        break
else:
    raise FileNotFoundError('Aucun fichier trouvé. Place ton CSV dans le dossier du notebook et renomme-le en bank_complaints.csv ou adapte la liste des fichiers candidats.')

print('Shape:', df.shape)
df.head()

## 2) Analyse exploratoire (EDA)

Regarde la structure, les valeurs manquantes, la distribution des labels, etc.

In [ ]:
# Aperçu rapide
print('Colonnes:', df.columns.tolist())

# Inspecter les colonnes texte et label communes
# Cherche des colonnes typiques: 'Complaint', 'Product', 'Issue', 'Description', 'Customer complaint', 'Complaint_ID'
text_cols = [c for c in df.columns if any(k in c.lower() for k in ['complaint','desc','description','narrative','text','message','issue'])]
label_cols = [c for c in df.columns if any(k in c.lower() for k in ['product','category','type','label'])]

print('Text columns candidates:', text_cols)
print('Label columns candidates:', label_cols)

# Si dataset a colonnes connues utilisées souvent sur Kaggle:
# 'Complaint ID', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Date received', 'Company'
if 'Consumer complaint narrative' in df.columns:
    text_col = 'Consumer complaint narrative'
elif len(text_cols) > 0:
    text_col = text_cols[0]
else:
    raise ValueError('Impossible de détecter automatiquement la colonne texte. Indique manuellement quelle colonne contient le texte.')

if 'Product' in df.columns:
    label_col = 'Product'
elif len(label_cols) > 0:
    label_col = label_cols[0]
else:
    # si pas d'étiquette, on peut créer une étiquette (ex: Customer complained -> binary) ou regrouper sur 'Issue'
    raise ValueError('Impossible de détecter automatiquement la colonne label. Indique manuellement la colonne à utiliser comme label.')

print('Using text column:', text_col)
print('Using label column:', label_col)

# Aperçu
display(df[[text_col, label_col]].head(10))

# Valeurs manquantes
print('\nMissing values in text:', df[text_col].isna().sum())
print('Missing values in label:', df[label_col].isna().sum())

# Distribution des labels
display(df[label_col].value_counts().head(20))

## 3) Prétraitement du texte

Fonctions de nettoyage : normalisation, suppression des URLs, numéros, ponctuation; tokenisation ; (optionnel) lemmatisation.

In [ ]:
# Prétraitement simple (français/anglais):
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except:
    nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))  # si dataset en français, remplace par 'french'
# pour le français: stopwords.words('french')
lemmatizer = WordNetLemmatizer()

def clean_text(text, lang='english'):
    if pd.isna(text):
        return ''
    # normalisation
    text = str(text).lower()
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    # tokenization simple + suppression stopwords
    tokens = nltk.word_tokenize(text)
    if lang == 'french':
        sw = set(stopwords.words('french'))
    else:
        sw = stop_words
    tokens = [t for t in tokens if t not in sw and len(t) > 1]
    # lemmatisation basique (WordNet pour anglais)
    if lang == 'english':
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Tester sur quelques lignes
df['__clean_text__'] = df[text_col].astype(str).apply(lambda x: clean_text(x, lang='english'))
df[['__clean_text__']].head()

## 4) Classification — TF-IDF + Naive Bayes (baseline)

On vectorise le texte avec TF-IDF et on entraîne un Multinomial Naive Bayes. Ensuite on affiche le rapport de classification.

In [ ]:
# Encode labels si nécessaire
le = LabelEncoder()
df = df.dropna(subset=['__clean_text__', label_col])
y = le.fit_transform(df[label_col])
X = df['__clean_text__']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer(max_df=0.9, min_df=5, ngram_range=(1,2))),
    ('clf', MultinomialNB())
])

pipe_nb.fit(X_train, y_train)
y_pred = pipe_nb.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion matrix shape:', cm.shape)

## 5) Modèle alternatif : Logistic Regression (avec GridSearch)

In [ ]:
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000))
])

params = {
    'tfidf__max_df': [0.9, 0.95],
    'tfidf__min_df': [3,5],
    'tfidf__ngram_range': [(1,1),(1,2)],
    'clf__C': [0.1, 1, 5]
}

grid = GridSearchCV(pipe_lr, params, cv=3, n_jobs=-1, scoring='f1_macro')
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
y_pred_lr = grid.predict(X_test)
print('Accuracy LR:', accuracy_score(y_test, y_pred_lr))
print('\nClassification report LR:\n', classification_report(y_test, y_pred_lr, target_names=le.classes_))

## 6) Visualisations

- Distribution des labels
- Matrice de confusion (heatmap alternative via matplotlib)


In [ ]:
# Distribution des labels
label_counts = pd.Series(y).map(lambda x: le.classes_[x]).value_counts()
plt.figure(figsize=(10,6))
label_counts.plot(kind='bar')
plt.title('Distribution des labels')
plt.xlabel('Label')
plt.ylabel('Nombre de réclamations')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de confusion (affichage simple)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion matrix (Naive Bayes)')
plt.xlabel('Predicted label index')
plt.ylabel('True label index')
plt.colorbar()
plt.tight_layout()
plt.show()

## 7) Sauvegarder le modèle et prochaines étapes

- Sauvegarder le pipeline avec joblib

```python
import joblib
joblib.dump(pipe_nb, 'complaint_classifier_nb.joblib')
```

- Prochaines améliorations :
  - Utiliser des embeddings (FastText, BERT/CamemBERT)
  - Faire du data augmentation pour classes déséquilibrées
  - Déployer via Flask/Streamlit


## Annexes — Conseils si ton dataset est en français

- Remplace `lang='english'` par `lang='french'` dans `clean_text`.
- Utilise les stopwords `stopwords.words('french')`.
- Pour la lemmatisation française, utilise spaCy (`fr_core_news_sm`) :

```bash
pip install spacy
python -m spacy download fr_core_news_sm
```

Puis dans le notebook :

```python
import spacy
nlp = spacy.load('fr_core_news_sm')
def clean_text_fr(text):
    doc = nlp(text)
    tokens = [t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct and len(t.text)>1]
    return ' '.join(tokens)
```
